# 17.8 4-bit 如何真的存成較小檔案？


七個四位元數若各放一個int8元素，仍會占七bytes，只限制數值範圍並沒有省掉容器。要真正縮小，可以把兩個四位元碼放進同一byte：一個占低四位，另一個占高四位。七個值需四bytes，最後半個byte作為補格；還原時必須知道原來只有七個，才能裁掉補格。前置是[bit數與容器大小](https://birdhackor.github.io/tiny-perceptron-vlm/17.6.html)及[刻度還原](https://birdhackor.github.io/tiny-perceptron-vlm/17.2.html)。

本教材允許signed值-8至7，先加8，得到0至15的儲存碼。例如-8變0，四位元寫成0000；-1變7，寫成0111。第一碼放低四位，第二碼往左移四位再合併，byte便是 `0111 0000`，十進位112。這套「加8」是我們的編碼約定，其他格式可能不同，拆碼時必須使用配套規則。

這是打包器可容納的範圍；本章量化函數實際採正負對稱的-7至7，會留下代表-8的儲存碼不用。兩者不能混為一談：打包不改整數值，量化才決定哪些整數格子能表示浮點權重。

圖由上方例子開始讀：原數字-8、-1各加8，映成四位元碼0000、0111。綠色虛線把第一碼指向byte右側的低四位，橘色實線把第二碼指向左側的高四位。交叉箭頭指示碼存入哪四位；拆回時仍依第一碼、第二碼還原原數字的順序。下方例子的0、1同理變成1000、1001，byte為10011000，也就是152。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/architecture_int4_packing.svg")))

In [ ]:
import torch
from tiny_perceptron.quantization import pack_int4, unpack_int4

q = torch.tensor([-8, -1, 0, 1, 7, 2, 3], dtype=torch.int8)
packed = pack_int4(q)
restored = unpack_int4(packed, q.shape)
print("原數字", q.tolist())
print("packed", packed.tolist(), "bytes", packed.numel() * packed.element_size())
print("拆回", restored.tolist())
assert torch.equal(q, restored)

`pack_int4`先攤平q，加8映成碼；把第二碼用 `<<4`左移，再與第一碼用位元OR合併。這裡兩碼占不同四位區域，OR就像把它們各自填進空位，不是浮點相加。拆回時 `&15`留下低四位，`>>4`移出高四位，再各減8。q.shape提供原形狀與元素數，避免將尾端補格當作真實權重。

第二行應為 `[112,152,175,139]`，占四bytes。第三組7、2加8得15、10，byte為 `10×16+15=175`；最後3加8得11，另一格補成碼8（還原為0），byte為 `8×16+11=139`。拆回輸出應與原七項完全相同，assert核對的是整數碼的無損打包，不代表先前浮點量化沒有誤差。

完整模型還要保存scale、原形狀與位元數；若線性層有bias，還要保存這份加在輸出上的偏移值，見[2.3的線性層配方](https://birdhackor.github.io/tiny-perceptron-vlm/2.3.html)。所以四bytes只是這段整數資料區（buffer）。存檔可能另有標頭，也就是記錄型別、形狀等資訊的部分，真正的檔案大小要另量。打包省儲存，但普通線性運算不能直接拿112等byte當權重乘輸入，必須先按格式拆解或使用懂該格式的專用運算。

正式4-bit模型的13個Linear共保存57,600 bytes整數碼、5,664 bytes逐列scale與2,560 bytes浮點bias，合計65,824 bytes的buffer。其他浮點參數還需102,912 bytes，兩邊相加才是[17.1](https://birdhackor.github.io/tiny-perceptron-vlm/17.1.html)的168,736 bytes。報告`storage.buffers`逐層列出這些數字，能核對刻度確實存下來了；它們是實際容器大小，不是先用參數個數除以二推估的理想值。

練習只在q最後加一項4。先預測仍占四bytes，最後byte由139變203，因為高四位不再補碼8，而是4加8得到12；拆回應有八項。重跑核對，說明原元素數為何是打包格式不可缺的一部分。
